In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install clearml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 37.7 MB/s eta 0:00:00


In [ ]:
import os
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    set_seed
)
from clearml import Task

In [ ]:
from tqdm import tqdm

In [ ]:
task = Task.init(
    project_name="cross-lingual-lm",
    task_name="swahili_retokenize_roberta_low_resource",
    task_type=Task.TaskTypes.training
)

logger = task.get_logger()

set_seed(42)

ClearML Task: created new task id=cdffac6c900943eba911e07a22127765


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


2026-04-20 13:33:18,485 - clearml.Task - INFO - Storing jupyter notebook directly as code
ClearML results page: https://app.clear.ml/projects/9bf855ce67034d8d9b44bf6ae4fbf457/experiments/cdffac6c900943eba911e07a22127765/output/log


In [ ]:
MODEL_PATH = "/content/drive/MyDrive/nlp_project/models/english_lm_roberta"

MAX_LENGTH = 128
BATCH_SIZE = 8
GRAD_ACCUM = 4

EPOCHS = 3
LR = 3e-5

SAMPLE_SIZE = 10000

OUTPUT_DIR = "./swahili-retokenize"


In [ ]:
task.connect({})

{}

In [ ]:
from datasets import load_dataset
dataset = load_dataset("ngusadeep/Swahili-Corpus-Dataset")["train"]

# Remove empty lines
dataset = dataset.filter(lambda x: len(x["text"].strip()) > 0)

# Limit dataset size (low-resource simulation)
dataset_dict = dataset.train_test_split(test_size=0.1, seed=42, shuffle=True)

raw_train_data = dataset_dict["train"]
eval_data = dataset_dict["test"]

train_data = raw_train_data.select(range(min(SAMPLE_SIZE, len(raw_train_data))))
eval_data = eval_data.select(range(min(3000, len(eval_data))))

print(f"Train size: {len(train_data)}")
print(f"Eval size:  {len(eval_data)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning:


The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.



README.md: 0.00B [00:00, ?B/s]

Swahili_Corpus_combined.txt:   0%|          | 0.00/253M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1693227 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1693227 [00:00<?, ? examples/s]

Train size: 10000
Eval size:  3000


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForMaskedLM.from_pretrained(MODEL_PATH)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

In [ ]:
def token_fragment_stats(dataset, tokenizer):
    stats = []

    for t in tqdm(dataset["text"]):
        tokens = tokenizer.tokenize(t)
        stats.append(len(tokens))

    return {
        "avg_len": sum(stats) / len(stats),
        "max_len": max(stats),
        "min_len": min(stats)
    }

In [ ]:
token_fragment_stats(train_data, tokenizer)

100%|██████████| 10000/10000 [00:04<00:00, 2190.80it/s]


{'avg_len': 56.5408, 'max_len': 910, 'min_len': 1}

## Train BPE on Swahili

In [ ]:
!mkdir sw_tokenizer_bpe

In [ ]:
from tokenizers import ByteLevelBPETokenizer

tokenizer = ByteLevelBPETokenizer()
texts = [x["text"] for x in train_data]

tokenizer.train_from_iterator(
    texts,
    vocab_size=12000,
    min_frequency=10,
    special_tokens=["<s>", "<pad>", "</s>", "<unk>", "<mask>"]
)

output = tokenizer.encode("habari gani")
print(output.ids)

tokenizer.save_model('/content/sw_tokenizer_bpe')

[4417, 850]


['/content/sw_tokenizer_bpe/vocab.json',
 '/content/sw_tokenizer_bpe/merges.txt']

## Check tokenizer saving

In [ ]:
!mkdir sw_tokenizer_final

In [ ]:
from tokenizers import Tokenizer
from transformers import RobertaTokenizerFast

tokenizer.save("/content/sw_tokenizer_bpe/tokenizer.json")


new_tokenizer = RobertaTokenizerFast(
    tokenizer_file="/content/sw_tokenizer_bpe/tokenizer.json",
    bos_token="<s>",
    eos_token="</s>",
    sep_token="</s>",
    cls_token="<s>",
    unk_token="<unk>",
    pad_token="<pad>",
    mask_token="<mask()",
    add_prefix_space=True
)

new_tokenizer.save_pretrained('/content/sw_tokenizer_final')

test_str = "habari gani"
encoded = new_tokenizer.encode(test_str, add_special_tokens=False)
print(f"IDs: {encoded}")
print(f"Tokens: {new_tokenizer.convert_ids_to_tokens(encoded)}")

IDs: [4417, 850]
Tokens: ['habari', 'Ġgani']


In [ ]:
import numpy as np

class FertilityMonitor:
    def __init__(self, old_tokenizer, new_tokenizer):
        self.old_tok = old_tokenizer
        self.new_tok = new_tokenizer

    def calculate_fertility(self, tokenizer, texts: list) -> float:
        total_tokens = 0
        total_words = 0

        for text in texts:
            words = text.split()
            if not words: continue

            # Токенизируем текст целиком для реалистичного замера
            tokens = tokenizer.encode(text, add_special_tokens=False)

            total_tokens += len(tokens)
            total_words += len(words)

        return total_tokens / total_words if total_words > 0 else 0

    def run_report(self, dataset, num_samples=1000):
        # Берем выборку текстов
        samples = dataset.select(range(min(num_samples, len(dataset))))["text"]

        old_f = self.calculate_fertility(self.old_tok, samples)
        new_f = self.calculate_fertility(self.new_tok, samples)

        improvement = ((old_f - new_f) / old_f) * 100

        print(f"Old Tokenizer (RoBERTa): {old_f:.3f} tokens/word")
        print(f"New Tokenizer (Swahili): {new_f:.3f} tokens/word")
        print(f"Efficiency Increase:     {improvement:.2f}%")


In [ ]:
from transformers import RobertaTokenizerFast, RobertaForMaskedLM

MODEL_PATH = "/content/drive/MyDrive/nlp_project/models/english_lm_roberta"

old_tokenizer = RobertaTokenizerFast.from_pretrained(MODEL_PATH)

model = RobertaForMaskedLM.from_pretrained(MODEL_PATH)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Check old (EN) and new (SWA) tokenizer

In [ ]:
print(f"Old vocab size: {len(old_tokenizer)}")
print(f"New vocab size: {len(new_tokenizer)}")

test_word = "habari"
print(f"Old: {old_tokenizer.tokenize(test_word)}")
print(f"New: {new_tokenizer.tokenize(test_word)}")

Old vocab size: 50265
New vocab size: 6109
Old: ['h', 'a', 'b', 'a', 'r', 'i']
New: ['habari']


In [ ]:
monitor = FertilityMonitor(old_tokenizer, new_tokenizer)
monitor.run_report(eval_data)

Token indices sequence length is longer than the specified maximum sequence length for this model (652 > 512). Running this sequence through the model will result in indexing errors


Old Tokenizer (RoBERTa): 5.380 tokens/word
New Tokenizer (Swahili): 1.433 tokens/word
Efficiency Increase:     73.36%


## Map new tokenizer to old one

In [ ]:
def build_mapping(old_tokenizer, new_tokenizer):
    old_vocab = old_tokenizer.get_vocab()
    mapping = {}
    unk_id = old_tokenizer.unk_token_id

    for token, new_id in new_tokenizer.get_vocab().items():
        # Заменяем специальный символ BPE (Ġ) на пробел для корректной токенизации старым токенайзером
        clean_token = token.replace('Ġ', ' ').strip() if 'Ġ' in token else token

        # Токенизируем новый токен старым токенайзером
        sub_pieces = old_tokenizer.tokenize(clean_token)

        valid_ids = [
            old_vocab[p] for p in sub_pieces
            if p in old_vocab
        ]

        if not valid_ids:
            mapping[new_id] = [(unk_id, 1.0)]
        else:
            weight = 1.0 / len(valid_ids)
            mapping[new_id] = [(idx, weight) for idx in valid_ids]

    return mapping

In [ ]:
import torch

@torch.no_grad()
def remap_embeddings(model, old_tokenizer, new_tokenizer, mapping):
    # Store old weights
    new_vocab_size = len(new_tokenizer)
    old_embeddings = model.get_input_embeddings()
    old_weights = old_embeddings.weight.data.clone()
    embedding_dim = old_weights.size(1)
    device = old_weights.device
    dtype = old_weights.dtype

    # Resize model for new embeddings
    model.resize_token_embeddings(new_vocab_size)
    new_embeddings = model.get_input_embeddings()

    updated_weights = torch.zeros((new_vocab_size, embedding_dim), device=device, dtype=dtype)

    # Transfer knowledge (Weighted Average)
    for new_id, source_list in mapping.items():
        if new_id >= new_vocab_size:
            continue

        for old_id, weight in source_list:
            updated_weights[new_id] += old_weights[old_id] * weight

    new_embeddings.weight.data.copy_(updated_weights)

    model.tie_weights()

    print(f"Successfully remapped {new_vocab_size} embeddings.")

In [ ]:
mapping = build_mapping(old_tokenizer, new_tokenizer)


In [ ]:
model = RobertaForMaskedLM.from_pretrained(MODEL_PATH)

remap_embeddings(
    model,
    old_tokenizer,
    new_tokenizer,
    mapping
)

# 4. Проверяем tie_weights (для RoBERTa это критично)
model.tie_weights()

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Successfully remapped 6109 embeddings.


In [ ]:
token_fragment_stats(train_data, new_tokenizer)

100%|██████████| 10000/10000 [00:02<00:00, 4123.36it/s]


{'avg_len': 31.5892, 'max_len': 568, 'min_len': 1}

In [ ]:
# 1. Проверяем, что в токенайзере всё на месте
print(f"Tokenizer mask ID: {new_tokenizer.mask_token_id}")

# 2. У RoBERTa эти поля в конфиге называются именно так:
model.config.pad_token_id = new_tokenizer.pad_token_id
model.config.bos_token_id = new_tokenizer.bos_token_id
model.config.eos_token_id = new_tokenizer.eos_token_id

# 3. Если хочешь сохранить ID маски в конфиге для истории (хотя модель его не использует напрямую через конфиг):
model.config.mask_token_id = new_tokenizer.mask_token_id # Это вызовет твою ошибку

Tokenizer mask ID: 6108


In [ ]:
model.config.mask_token_id

6108

In [ ]:
# def tokenize_function(examples):
#     return new_tokenizer(
#         examples["text"],
#         truncation=True,
#         padding="max_length",
#         max_length=128
#     )
# train_dataset = train_data.map(tokenize_function)
# eval_dataset = eval_data.map(tokenize_function)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

In [ ]:
def group_texts(examples, block_size=128):
    # Собираем все тексты в одну кучу
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])

    # Отбрасываем остаток, который меньше block_size
    total_length = (total_length // block_size) * block_size

    # Режем на равные куски по block_size
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }

    # Для MLM нам нужны labels (коллатор их сделает сам, но можно подготовить)
    result["labels"] = result["input_ids"].copy()
    return result

tokenized_datasets = train_data.map(
    lambda x: new_tokenizer(x["text"], truncation=True, add_special_tokens=True),
    batched=True,
    remove_columns=["text"]
)

# 2. Группируем в блоки по 128
lm_dataset = tokenized_datasets.map(
    group_texts,
    batched=True,
    fn_kwargs={"block_size": 128}
)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

## Stage 1

In [ ]:
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=new_tokenizer,
    mlm=True,
    mlm_probability=0.15
)

# Заморозка и разморозка
model.requires_grad_(False)
model.get_input_embeddings().weight.requires_grad = True
model.get_output_embeddings().weight.requires_grad = True

args_stage1 = TrainingArguments(
    output_dir="./results_stage1",
    per_device_train_batch_size=32,
    learning_rate=2e-4,
    num_train_epochs=10,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="epoch",
    warmup_ratio=0.1,
    fp16=True
)

trainer_stage1 = Trainer(
    model=model,
    args=args_stage1,
    train_dataset=lm_dataset,
    data_collator=data_collator,
    optimizers=(torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4), None)
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
trainer_stage1.train()

Step,Training Loss
50,8.787640
100,8.031337
150,7.642367
200,7.567906
250,7.501566
300,7.480894
350,7.470139
400,7.442443
450,7.399292
500,7.399386


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=770, training_loss=7.573475587522829, metrics={'train_runtime': 383.3833, 'train_samples_per_second': 64.244, 'train_steps_per_second': 2.008, 'total_flos': 1620221842414080.0, 'train_loss': 7.573475587522829, 'epoch': 10.0})

# Evaluate 1 stage

In [ ]:
import math
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
def eval_ppl(MODEL_PATH, dataset):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    model = AutoModelForMaskedLM.from_pretrained(MODEL_PATH)
    model.to(device)
    model.eval()

    collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=True,
        mlm_probability=0.15
    )

    args = TrainingArguments(
        output_dir="./eval_en_tmp",
        per_device_eval_batch_size=BATCH_SIZE,
        report_to=[]
    )

    trainer = Trainer(
        model=model,
        args=args,
        eval_dataset=dataset,
        data_collator=collator
    )


    torch.manual_seed(42)
    metrics = trainer.evaluate()

    loss = metrics["eval_loss"]
    perplexity = math.exp(loss)

    print(f"Loss: {loss:.4f}")
    print(f"Perplexity: {perplexity:.2f}")

In [ ]:
eval_tokenized = eval_data.map(
    lambda x: new_tokenizer(x["text"], truncation=True, add_special_tokens=True),
    batched=True,
    remove_columns=["text"]
)

eval_lm_dataset = eval_tokenized.map(
    group_texts,
    batched=True,
    fn_kwargs={"block_size": 128}
)

In [ ]:
eval_ppl("/content/results_stage1/checkpoint-770", eval_lm_dataset)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Loss: 7.2824
Perplexity: 1454.41


# Stage 2

In [ ]:
for name, param in model.named_parameters():
    if "LayerNorm" in name or "word_embeddings" in name or "lm_head" in name:
        param.requires_grad = True
    # Опционально: разморозь последний слой трансформера (layer.11)
    if "layer.11" in name:
        param.requires_grad = True

args_stage2 = TrainingArguments(
    output_dir="./results_stage2",
    per_device_train_batch_size=32,
    learning_rate=5e-5,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="epoch",
    fp16=True,
    report_to="none"
)

trainer_stage2 = Trainer(
    model=model,
    args=args_stage2,
    train_dataset=lm_dataset,
    data_collator=data_collator,
    optimizers=(torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=5e-5), None)
    )

trainer_stage2.train()

Step,Training Loss
50,7.310114
100,7.232273
150,7.201677
200,7.195479
250,7.152989
300,7.157599
350,7.175019


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=385, training_loss=7.198407973252333, metrics={'train_runtime': 180.9173, 'train_samples_per_second': 68.07, 'train_steps_per_second': 2.128, 'total_flos': 810110921207040.0, 'train_loss': 7.198407973252333, 'epoch': 5.0})

In [ ]:
eval_ppl("/content/results_stage2/checkpoint-385", eval_lm_dataset)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Loss: 7.0815
Perplexity: 1189.76


In [ ]:
# 1. Заново токенизируем (или используем старый результат),
# убедившись, что truncation позволяет брать больше токенов
tokenized_datasets_wide = train_data.map(
    lambda x: new_tokenizer(x["text"], truncation=True, max_length=512, add_special_tokens=True),
    batched=True,
    remove_columns=["text"]
)

# 2. Группируем в блоки по 256
lm_dataset_256 = tokenized_datasets_wide.map(
    group_texts,
    batched=True,
    fn_kwargs={"block_size": 256}
)

# Сделай то же самое для валидационного набора
eval_lm_dataset_256 = eval_tokenized.map(
    group_texts,
    batched=True,
    fn_kwargs={"block_size": 256}
)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

In [ ]:
model.requires_grad_(True)

args_stage3 = TrainingArguments(
    output_dir="./results_stage3",
    per_device_train_batch_size=16,
    learning_rate=2e-5,
    num_train_epochs=5,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    warmup_steps=500,
    logging_steps=50,
    save_strategy="steps",
    save_steps=1000,
    fp16=True,
    report_to="none"
)

trainer_stage3 = Trainer(
    model=model,
    args=args_stage3,
    train_dataset=lm_dataset_256,
    data_collator=data_collator
)

trainer_stage3.train()

Step,Training Loss
50,7.189164
100,7.162100
150,7.151824
200,7.134327
250,7.122571
300,7.084194
350,7.058885


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=385, training_loss=7.121211302125609, metrics={'train_runtime': 131.9233, 'train_samples_per_second': 46.58, 'train_steps_per_second': 2.918, 'total_flos': 808466359856640.0, 'train_loss': 7.121211302125609, 'epoch': 5.0})

In [ ]:
eval_ppl("/content/results_stage3/checkpoint-385", eval_lm_dataset_256)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Loss: 6.9624
Perplexity: 1056.20


In [ ]:
from transformers import pipeline

fill_mask = pipeline("fill-mask", model='/content/results_stage3/checkpoint-385', tokenizer=new_tokenizer)

examples = [
    f"Habari za {new_tokenizer.mask_token}?",
    f"Mimi ni {new_tokenizer.mask_token}.",
    f"Jina langu ni {new_tokenizer.mask_token}."
]

for ex in examples:
    print(f"\nPrompt: {ex}")
    for res in fill_mask(ex):
        print(f"  {res['score']:.4f} -> {res['token_str']}")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]


Prompt: Habari za <mask()?
  0.0145 ->  na
  0.0142 ->  ya
  0.0115 ->  wa
  0.0099 ->  kwa
  0.0060 ->  za

Prompt: Mimi ni <mask().
  0.0132 ->  wa
  0.0098 ->  ya
  0.0086 ->  na
  0.0080 ->  kwa
  0.0037 ->  kuwa

Prompt: Jina langu ni <mask().
  0.0126 ->  wa
  0.0108 ->  ya
  0.0103 ->  na
  0.0073 ->  kwa
  0.0048 ->  la


In [ ]:
# 1. Замораживаем обратно верхние слои, оставляем только низ
model.requires_grad_(False)
for i in range(3): # Разморозим только первые 3 слоя
    model.roberta.encoder.layer[i].requires_grad_(True)
model.get_input_embeddings().weight.requires_grad = True
model.lm_head.requires_grad_(True)

args_low_resource = TrainingArguments(
    output_dir="./results_low_res",
    per_device_train_batch_size=16,
    learning_rate=5e-5,
    num_train_epochs=50,         # КРИТИЧНО: Увеличиваем количество проходов
    weight_decay=0.05,           # Увеличиваем регуляризацию
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=100,
    fp16=True
)

trainer_low_res = Trainer(
    model=model,
    args=args_low_resource,
    train_dataset=lm_dataset_256,
    data_collator=data_collator
)
trainer_low_res.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss
100,7.005321
200,6.971963
300,6.924013
400,6.872975
500,6.868749
600,6.832913
700,6.789445
800,6.704711
900,6.690794
1000,6.641924


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3850, training_loss=6.35079196682224, metrics={'train_runtime': 980.9377, 'train_samples_per_second': 62.644, 'train_steps_per_second': 3.925, 'total_flos': 8084663598566400.0, 'train_loss': 6.35079196682224, 'epoch': 50.0})

In [ ]:
eval_ppl("/content/results_low_res/checkpoint-3850", eval_lm_dataset_256)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Loss: 5.9166
Perplexity: 371.14


In [ ]:
fill_mask = pipeline("fill-mask", model="/content/results_low_res/checkpoint-3850", tokenizer=new_tokenizer)

test_prompts = [
    f"Habari za {new_tokenizer.mask_token}?",
    f"Mimi ni {new_tokenizer.mask_token}.",
    f"Jina langu ni {new_tokenizer.mask_token}."
]

for prompt in test_prompts:
    print(f"\n{prompt}")
    for res in fill_mask(prompt):
        print(f"  {res['score']:.4f} -> {res['token_str']}")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]


Habari za <mask()?
  0.0062 -> r
  0.0058 ->  na
  0.0048 ->  kuwa
  0.0044 ->  ya
  0.0043 ->  ku

Mimi ni <mask().
  0.0133 -> r
  0.0096 -> m
  0.0083 ->  m
  0.0082 ->  na
  0.0078 ->  ya

Jina langu ni <mask().
  0.0160 ->  na
  0.0160 ->  
  0.0138 ->  wa
  0.0122 ->  kwa
  0.0118 ->  ya


In [ ]:
output_dir = 'roberta_swahili_retokenize'

In [ ]:
trainer_low_res.save_model(output_dir)

new_tokenizer.save_pretrained(output_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('roberta_swahili_retokenize/tokenizer_config.json',
 'roberta_swahili_retokenize/tokenizer.json')

In [ ]:
import shutil
import os

shutil.copytree(output_dir, '/content/drive/MyDrive/nlp_project/models/roberta_swahili_retokenize', dirs_exist_ok=True)

'/content/drive/MyDrive/nlp_project/models/roberta_swahili_retokenize'

# Evaluate on EN PPL

In [ ]:
DATASET_NAME = "wikitext"
DATASET_CONFIG = "wikitext-103-raw-v1"

MAX_LENGTH = 128
BATCH_SIZE = 8
EVAL_SAMPLES = 3000

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
dataset = load_dataset(DATASET_NAME, DATASET_CONFIG, split="test")

dataset = dataset.filter(lambda x: x["text"] and len(x["text"].strip()) > 0)
dataset = dataset.select(range(min(EVAL_SAMPLES, len(dataset))))

dataset

README.md: 0.00B [00:00, ?B/s]

wikitext-103-raw-v1/test-00000-of-00001.(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00000-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00001-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

wikitext-103-raw-v1/validation-00000-of-(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1801350 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4358 [00:00<?, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 2891
})

In [ ]:
tokenized_dataset = dataset.map(
    lambda x: new_tokenizer(x["text"], truncation=True, max_length=512, add_special_tokens=True),
    batched=True,
    remove_columns=["text"]
)

eval_en = tokenized_dataset.map(
    group_texts,
    batched=True,
    fn_kwargs={"block_size": 256}
)


Map:   0%|          | 0/2891 [00:00<?, ? examples/s]

Map:   0%|          | 0/2891 [00:00<?, ? examples/s]

In [ ]:
eval_ppl("/content/drive/MyDrive/nlp_project/models/roberta_swahili_retokenize", eval_en)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Loss: 5.9421
Perplexity: 380.75


# Evaluate on Swahili PPL

In [ ]:
eval_ppl("/content/drive/MyDrive/nlp_project/models/roberta_swahili_retokenize", eval_lm_dataset_256)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Loss: 5.9166
Perplexity: 371.14


In [ ]:
from transformers import pipeline

fill_mask = pipeline("fill-mask", model=model, tokenizer=tokenizer)

examples = [
    f"Habari za {tokenizer.mask_token}?",
    f"Mimi ni {tokenizer.mask_token}.",
    f"Jina langu ni {tokenizer.mask_token}."
]

for ex in examples:
    print(f"\nPrompt: {ex}")
    for res in fill_mask(ex):
        print(f"  {res['score']:.4f} -> {res['token_str']}")

In [ ]:
task.close()